# Customer Churn - Telecom
---

### CRISP-DM Methodology
This project follows the CRISP-DM (*Cross-Industry Standard Process for Data Mining*) framework applied to **Customer Retention & Churn Prediction**:
| **Stage** | **Objective** | **Methodological Execution** |
| :--- | :--- | :--- |
| **1. Business Understanding** | Mitigate revenue loss by identifying at-risk customers. | • **Target Definition**: Binary Classification (Churn: Yes/No).<br>• **KPIs**: Maximize **Lift** in retention campaigns & Revenue Saved vs. Cost. |
| **2. Data Understanding** | Detect patterns of friction and dissatisfaction. | • **EDA**: Distribution analysis (Detect Imbalance).<br>• **Hypothesis Testing**: Correlation Matrix & Independence Tests (Chi-Square). |
| **3. Data Preparation** | Construct a robust dataset for parametric modeling. | • **Scaling**: Standardization (Z-score) for coefficient comparability.<br>• **Encoding**: One-Hot Encoding for nominal variables.<br>• **Splitting**: Stratified Train/Test Split to preserve class ratio. |
| **4. Modeling** | Estimate Churn Probability | • **Algorithms**: Logistic Regression, LinearSVC, KNN, Random Florest, XGBoost, LightGBM.<br>• **Inference**: Analyze **Odds Ratios** to determine feature elasticity. |
| **5. Evaluation** | Assess model reliability and financial impact. | • **Discrimination**: AUC-ROC & F1-Score (Threshold Tuning).<br>• **Calibration**: Probability Calibration Curve (Reliability Diagram). |
| **6. Deployment** | Integrate insights into the CRM lifecycle. | • **Deliverable**: "High-Risk" Customer List for Marketing Squad.<br>• **Artifact**: Serialize model (`joblib`) for batch inference. |

---

### Installs:

In [0]:
%%capture
%pip install -r '../requirements.txt'
# Command to restart the kernel and update the installed libraries
%restart_python

### Imports:

In [0]:
# ======================================================== #
# Standard library
# ======================================================== #
import joblib

# ======================================================== #
# Third-party libraries - data analysis and visualization
# ======================================================== #
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ======================================================== #
# Third-party libraries - modeling and preprocessing
# ======================================================== #
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
import optuna
import warnings 
warnings.filterwarnings('ignore')


In [0]:
# SRC/ Functions Utils:
import sys
sys.path.append('../src')
from visualization import GraphicsData
from utils import EDATest
from modeling_utils import DtypeOptimizer, FeatureEngineer, classification_kfold_cv, RecursiveFeatureEliminator

### Dev objects

### Load the data

In [0]:
df = pd.read_csv('../data/ChurnData.csv')

### Verify successful load with some randomly selected records


In [0]:
df.sample(9)

In [0]:
df.head()

In [0]:
df.info()


### 3. Data Preparation

In [0]:
GraphicsData(data = df).plot_target_analysis(target_col='churn', colors=['#1abc9c', '#ff6b6b'])

##### Train and Test Data Split
---

- Before starting the data preprocessing and modeling process, the dataset will be split into **Training** and **Test** sets.

The main goal is to prevent data leakage, ensuring that all statistical information, outlier handling, transformation decisions, and feature engineering strategies are derived exclusively from the training data.

> This approach preserves the integrity of the validation process and ensures that performance metrics reflect the model’s true generalization capability, rather than contamination from information in the test set.

- The `stratify` parameter will be applied in the `train_test_split` procedure.

Since churn prediction is a binary classification problem, keeping the original class proportion in both subsets is statistically recommended and, in practice, helps avoid evaluation bias.

> Stratified sampling preserves the prior distribution of the target variable, reducing distortions in class balance that could bias model training, decision-threshold calibration, and metrics such as Recall, Precision, and ROC-AUC.

---


In [0]:
X = df.drop(columns = ['churn'])
y = df['churn'].astype('int8').copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y,  random_state = 33)

In [0]:
# Checking the proportions of the target variable
print(f'Shape of training: {X_train.shape}')
print(f'Shape of test: {X_test.shape}')

print('\n--- Churn Rate (Stratify Validation) ---')
print(f'Original: {df['churn'].mean():.2%}')
print(f'Train:    {y_train.mean():.2%}')
print(f'Test:    {y_test.mean():.2%}')

##### Construction of Data Preparation Pipelines
---


- For the data preparation stage, **two distinct pipelines** were developed: one designed for **linear models** and the other for **tree-based models**.


> This separation was adopted because each family of algorithms has its own preprocessing requirements, especially with regard to **scaling numerical variables** and **encoding categorical variables**.


- As discussed in the EDA, the initial decision was to remove only the variables **`equip`** and **`wireless`**, since they showed high correlation with **`equipmon`** and **`wiremon`**, respectively.


- The other highly correlated variables were deliberately retained at this initial stage.


> This decision allows the modeling process to empirically evaluate how different algorithms handle **multicollinearity**, **informational redundancy**, and the possible **marginal predictive gain** associated with these variables.


- In both pipelines, the data preparation flow followed the same general structure:


- **Variable type optimization**, with the objective of ensuring structural consistency, reducing memory usage, and adapting the data to the computational requirements of the algorithms.

- **Feature engineering**, with the creation of new derived variables based on findings from the exploratory analysis and domain knowledge.

- **Model-family-specific preprocessing**, respecting the technical particularities of each modeling approach and ensuring compatibility between the transformed data and the algorithms used.


---


##### Checking the numeric variables

In [0]:
pipeline_fg = Pipeline(
    steps = [
        ('types_vars', DtypeOptimizer()),
        ('fg', FeatureEngineer())
    ]
)
train_set_fg = pipeline_fg.fit_transform(X_train)
train_set_fg.head()

In [0]:
GraphicsData(train_set_fg.select_dtypes(include = ['float32'])).numerical_histograms()

#### Pre-processor for Linear Models

In [0]:
# Columns
one_hot_cols = ['ed', 'custcat', 'tenure_band',]

num_cols = ['tenure', 'age', 'address', 'employ', 'tollmon', 'equipmon', 'share_cardmon', 'share_tollmon', 'nonzero_count',
'tenure_x_internet', 'tenure_x_wireless', 'tenure_x_ebill', 'tenure_x_equip', 'total_services', 'stability_index_raw','ed_income_bucket','toxic_score']
 
skew_cols = [
    'income', 'longmon','cardmon','wiremon', 'total_spend','share_wiremon','share_longmon', 'share_equipmon', 'total_spend_per_tenure',
    'income_per_age', 'income_per_ed',
]

bin_cols = ['callcard', 'voice', 'pager', 'internet', 'callwait', 'confer', 'ebill','flag_used_longmon', 'flag_used_tollmon', 'flag_used_equipmon', 'flag_used_cardmon', 'flag_used_wiremon', 'legacy_bundle', 'legacy_bundle_no_internet', 'young_low_tenure',
'old_upper_tenure']

drop_cols = ['equip', 'wireless']

# --------------------------------------------------------------------------------------------------------------------------------------#
# Pipelines

# Numerical columns with asymmetry
num_log = Pipeline(
    steps = [
        ('log', FunctionTransformer(np.log1p, feature_names_out = 'one-to-one')),
        ('std', StandardScaler()),
    ]
)

# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first', 
    sparse_output = False, 
    handle_unknown = 'ignore', 
    feature_name_combiner = 'concat', 
    dtype = np.int8
)

linear_preprocess = ColumnTransformer(
    transformers = [
        ('num_cols', StandardScaler(),num_cols), 
        ('skew_cols', num_log, skew_cols),
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('bin_cols', 'passthrough', bin_cols),
        ('drop_cols', 'drop', drop_cols)
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')


linear_pipeline = Pipeline(
    steps = [
       ('types_vars', DtypeOptimizer()), 
       ('feature_eg', FeatureEngineer()),
       ('linear_preprocess', linear_preprocess)
    ]
)

X_train_prepared_linear = linear_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_linear.head()

#### Pre-processor for Tree Models

In [0]:

# Colunas
one_hot_cols = ['custcat']
ord_cols = ['tenure_band']
drop_cols = ['equip', 'wireless'] 

# One Hot Config
one_hot_encoder = OneHotEncoder(
    drop = 'first',
    sparse_output=False,
    handle_unknown = 'ignore',
    feature_name_combiner = 'concat',
    dtype = np.int8
)

# Ordinal Config
# Order (same as pd.cut labels)
labels = ['0-3', '4-6', '7-12', '13-24', '25-48', '49+'] 
ordinal_encoder = OrdinalEncoder(
    categories = [labels],                 
    dtype = np.int8,
    handle_unknown='use_encoded_value', 
    unknown_value = -1
)

tree_preprocess = ColumnTransformer(
    transformers=[
        ('one_hot_cols', one_hot_encoder, one_hot_cols),
        ('ord_cols', ordinal_encoder, ord_cols),
        ('drop_cols', 'drop', drop_cols),
    ],
    remainder = 'passthrough',
    verbose_feature_names_out = False,
).set_output(transform = 'pandas')

tree_pipeline = Pipeline(steps=[
    ('types_vars', DtypeOptimizer()),
    ('feature_eg', FeatureEngineer()),
    ('tree_preprocess', tree_preprocess),
])

X_train_prepared_tree = tree_pipeline.fit_transform(X_train)

In [0]:
X_train_prepared_tree.head()

In [0]:
X_train_prepared_tree = X_train_prepared_tree.drop(columns = ['toxic_score', 'ed_income_bucket', 'income_per_ed','flag_used_longmon','flag_used_tollmon', 'flag_used_equipmon', 'flag_used_cardmon', 'flag_used_wiremon', 'tenure', ])

In [0]:
xx

### 4. Modeling
---

##### Model Evaluation and Selection Strategy
---


- The primary metric defined for this project will be **AUC-ROC**.


> Since the problem involves **imbalanced classes** and the central objective is to develop a model capable of estimating the **probability of churn**, AUC-ROC is appropriate because it provides a comprehensive view of the model's discriminative ability between the two classes.


- As secondary metrics, **F1-score** and **recall for the churn class** will also be monitored.


> The **F1-score** will be used to evaluate the balance between **precision** and **recall**, while **recall for the churn class** will receive special attention, since correctly identifying customers at higher risk of churn is essential for guiding retention actions.


> This choice is strategically relevant, since **retaining a customer** through engagement and loyalty campaigns is approximately **five times less costly** than **acquiring a new customer**.


- Initial training will be conducted using **cross-validation**, with the objective of increasing the robustness of model evaluation at this stage of the project.


> **5 folds** will be used, and the selection of the best model will consider not only the **highest mean AUC-ROC**, but also the **lowest variability** among the results observed across the different folds, seeking consistent performance and greater generalization capability.


- Simpler models, such as **Logistic Regression** and **Decision Tree Classifier**, will be tested.


> This choice prioritizes solutions with **lower complexity** and **greater interpretability**, which becomes especially important given a relatively small dataset with only **200 records**, reducing the risk of **overfitting**.


- And more robust models, such as **Linear SVC** and **LightGBM**, will also be evaluated.


> The objective of this comparison is to verify whether the structure of the data benefits from algorithms with **greater modeling capacity**, making it possible to assess potential performance gains in relation to simpler approaches.


---


#### Cross-Validation

In [0]:
linear_models = {
    'Logistic Regression': LogisticRegression(),
    'Linear SVC': LinearSVC(),
    'KNeighbors Classifier': KNeighborsClassifier(),
}

eval_linear = classification_kfold_cv(
    models = linear_models,
    X_train = X_train_prepared_linear,
    y_train = y_train,
    n_folds = 5,
    scoring = 'roc_auc'
)
eval_linear

In [0]:
tree_models = {
    'Decision Tree ': DecisionTreeClassifier(),
    'Random Forest ': RandomForestClassifier(),
    'CatBoost ': CatBoostClassifier(verbose = False),# verbose = False >> This is just a parameter that disables verbose e in CatBoost.
    'XGB ': XGBClassifier(),
    'LightGBM ': LGBMClassifier(verbosity = -1, ), # verbosity = -1 >> This is just a parameter that disables verbose e in LGBTM.
}
eval_tree = classification_kfold_cv(
    models = tree_models,
    X_train = X_train_prepared_tree,  
    y_train = y_train,
    n_folds = 5,
    
)
eval_tree

In [0]:
df_scores = pd.concat([eval_linear, eval_tree], ignore_index = True)
df_scores = df_scores[['avg_val_score', 'val_score_std', 'model']]


#### Models Scores 

In [0]:
GraphicsData(df_scores.sort_values('avg_val_score')).models_performance_barplots(models_col = 'model', figsize_per_row = 7,)

#### Initial Results and Direction for Feature Selection
---


- The results obtained so far can be considered **satisfactory**.


> The **LightGBM** model achieved an **AUC-ROC of 84.78%** and a **variance of 0.0458**, which is a relatively low value, indicating good stability across the evaluated folds.


- Despite the satisfactory performance, there are still important limitations in the dataset that must be considered at this stage.


> Among the main points of attention are the **small number of observations**, with only **200 records**, as well as the presence of **skewed variables** and variables with **low statistical power**, factors that may restrict the model's generalization capacity.


- Even so, several variables demonstrated good separation ability between **churners** and **non-churners**.


> During the statistical tests performed in the exploratory analysis stage, it was observed that some of these variables, in addition to showing **statistical significance**, also had a relevant impact on the distinction between the groups.


- This behavior was also reflected in the initial training results, especially in the performance of **LightGBM**.


> As a **tree-based** model, its structure tends to handle nonlinear relationships, interactions between variables, and heterogeneous patterns in the data efficiently.


- In light of this scenario, **LightGBM** will be used in the **feature selection** stage.


> The objective will be to **simplify the model** and **remove redundant variables**, with the expectation of further improving predictive performance while also making the solution more **compact** and **interpretable**.


---



#### Feature Selection 
---


- **RFECV (Recursive Feature Elimination with Cross Validation)** will be used to select the most relevant variables for the model and remove redundant features or those with lower contribution.


> This approach allows the feature selection process to be conducted more robustly, combining recursive feature elimination with cross-validation to identify the configuration that best supports model performance.


- In this way, the goal is to obtain a **simpler**, more **compact**, and more **interpretable** solution, without significant loss in predictive performance.


> The expectation is to reduce the complexity of the feature space, preserve the variables with the strongest informational signal, and improve the efficiency of the modeling process while maintaining consistent performance across evaluations.


---


In [0]:
selector = RecursiveFeatureEliminator(
    estimator = LGBMClassifier(verbosity = -1,),
    scoring = 'roc_auc',
    n_folds = 5
)

selector.fit(X_train_prepared_tree, y_train)

X_train_selected = selector.transform(X_train_prepared_tree)

print(selector.selected_features_)
print(f'\n\nInitial Features: {len(X_train_prepared_tree.columns)}')
print(f'\nRemainder Features: {len(X_train_selected.columns)}')

In [0]:
X_train_selected.head()

##### Feature Selection Results with RFE
---


- The results obtained with the **Recursive Feature Eliminator** can be considered **satisfactory**.


> Of the initial **49 features**, **34 were retained** after the selection process, indicating a reduction in the feature space without losing the most relevant attributes for modeling.


- Among the selected variables, **`tenure`** and its derived features stand out.


> This result suggests that factors associated with the **stability of the customer relationship** still play an important role in explaining churn at the institution.


---


#### Hyperparameter Optimization with Optuna
---


- In the **hyperparameter tuning** stage, **Optuna** will be used with the goal of making the hyperparameter optimization process more efficient than exhaustive methods such as **GridSearch**.


> This approach enables a more targeted exploration of the search space, prioritizing hyperparameter combinations with greater performance potential instead of evaluating all possibilities uniformly.


- Instead of testing combinations in a purely random way, **Optuna** uses sampling algorithms such as **TPE (Tree-structured Parzen Estimator)**.


> This mechanism takes into account the history of previous **trials** to identify more promising and less promising regions of the hyperparameter space, making the search process more adaptive and efficient.


- Considering the particular characteristics of this dataset, the definition of the search space was guided by a balance between hyperparameters that help mitigate limitations associated with the **small number of observations**.


> This strategy aims to control model complexity, reducing sensitivity to sample variation and minimizing the risk of excessive fitting to the training data.

---


In [0]:
def objective(trial):
    params = {
        'is_unbalance': True,
        'boosting_type': 'gbdt',
        'subsample_freq': 1,
        'objective': 'binary',
        'metric': 'auc',
        'importance_type': 'gain',
        'max_depth': 2,
        'num_leaves': 4,
        #'max_depth': trial.suggest_int('max_depth', 2, 8),
        #'num_leaves': trial.suggest_int('num_leaves', 2, 8),
        'min_child_samples': trial.suggest_int('min_child_samples', 15, 30),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log = True),# 1e-2, 8e-2
        'n_estimators': trial.suggest_int('n_estimators', 150, 300, step = 50),
        'subsample': trial.suggest_float('subsample', 0.8, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.8),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 0.01),
        'max_bin': trial.suggest_int('max_bin', 70, 90),
        'min_data_in_bin': trial.suggest_int('min_data_in_bin', 5, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 0.1, log = True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 0.1, log = True),
        'verbosity': -1,
        'random_state': 33,
        'n_jobs': 1

    }

    model = LGBMClassifier(**params)

    cv = StratifiedKFold(
        n_splits = 5,
        shuffle = True,
        random_state = 33
    )

    scores = cross_val_score(
        model,
        X_train_selected,
        y_train,
        scoring = 'roc_auc',
        cv = cv,
        n_jobs = -1
    ) 

    trial.set_user_attr('auc_std', float(np.std(scores)))
    return float(np.mean(scores))


study = optuna.create_study(
    direction='maximize',
    study_name='lgbm_optuna_auc'
)

study.optimize(objective, n_trials = 100)

best_params = study.best_params
best_score = study.best_value
best_std = study.best_trial.user_attrs['auc_std']

print('\nBest Hyperparameters:', best_params)
print('Best Mean ROC-AUC:', best_score)
print('Best ROC-AUC Std:', best_std)

#### Final Training 

In [0]:
best_params = {
    'is_unbalance': True,
    'boosting_type': 'gbdt',
    'subsample_freq': 1,
    'objective': 'binary',
    'metric': 'auc',
    'importance_type': 'gain',
    'max_depth': 2,
    'verbosity': -1,
    'random_state': 33,
    'n_jobs': 1,
    'num_leaves': 4, 
    'min_child_samples': 30, 'learning_rate': 0.01688305288629162, 'n_estimators': 300, 'subsample': 0.811973759959785, 'colsample_bytree': 0.7141390706896851, 'min_split_gain': 0.000315526631760518, 'max_bin': 71, 'min_data_in_bin': 6, 'reg_alpha': 0.017392581716417145, 'reg_lambda': 0.0554588027302538
}

final_model = LGBMClassifier(** best_params)
final_model.fit(X_train_selected, y_train)

#### Testset Preparation

In [0]:
X_test_prepared_tree = tree_pipeline.transform(X_test)
X_test_prepared_tree = X_test_prepared_tree.drop(columns =  ['toxic_score', 'ed_income_bucket', 'income_per_ed','flag_used_longmon','flag_used_tollmon', 'flag_used_equipmon', 'flag_used_cardmon', 'flag_used_wiremon', 'tenure', ])
X_test_selected =  selector.transform(X_test_prepared_tree)
X_test_selected.head()

#### Prediction

In [0]:
y_pred = final_model.predict(X_test_selected) 
y_prob = final_model.predict_proba(X_test_selected)[:, 1]

In [0]:
from sklearn.metrics import roc_auc_score

auc_roc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

In [0]:
from sklearn.metrics import roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_pred = final_model.predict(X_test_selected)
y_prob = final_model.predict_proba(X_test_selected)[:, 1]

auc_roc = roc_auc_score(y_test, y_prob)
print(f"AUC-ROC: {auc_roc:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("Matriz de Confusão:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Não Churn', 'Churn'])
disp.plot(cmap='Blues')
plt.title('Matriz de Confusão')
plt.show()


### 5. Evaluation:

#### Key Observations:
---

#### 1. Technical Performance

  - **Explanatory Power (Real R² Score): 0.8844**

  - The model explains **88.4% of the variability** in CO2 emissions using the "Real World" scale (g/km). In the transformed mathematical space (Yeo-Johnson), the fit is even higher (**0.91**), confirming that the non-linear approach successfully captured the physical behavior of the data. This is a significant improvement over the simple univariate model (~0.80).

  - **Margin of Error (MAE): 13.03**

  - The **Mean Absolute Error** indicates that, on average, our predictions deviate by only **13.03 g/km** from the actual value. For a business context where emissions range up to 450 g/km, this represents a very high precision level (approx. 5% relative error), allowing for reliable carbon footprint estimation.

  - **Sensitivity to Large Errors (RMSE vs MAE):**

  * The **RMSE (20.87)** is controlled relative to the MAE (13.03). The gap of ~7.8 points is healthy. It indicates that while there are outliers (likely high-performance sports cars or heavy vehicles), the **Yeo-Johnson transformation** successfully mitigated the extreme penalties that usually distort linear models.
---

#### 2. Model Interpretation

  - The model utilizes a **Power Law** approach (Yeo-Johnson) rather than a simple straight line:

  - **Feature Dominance (Standardized Coefficients):**
  - **Fuel Consumption (Coefficient ~0.71):** This is the dominant driver. The stability analysis showed a variation of only **1.92%** (CV) for this weight, proving it is the most reliable predictor.
  - **Engine Size (Coefficient ~0.27):** This acts as a secondary adjustment factor. Even with a 0.82 correlation to fuel, the model successfully isolated its unique contribution with high stability (CV ~4.05%).

  - **The "Curved" Surface:**
  - Unlike a rigid linear equation, the model projects a **curved surface**. This means it understands that "efficiency" changes as engines get bigger. Physically, this represents the diminishing returns of combustion efficiency in larger engines, providing a much more realistic simulation than a simple linear slope.
---

#### 3. **Conclusion:**

- The Multiple Regression Model with Power Transformation represents the "State-of-the-Art" for this dataset.

- **Strengths:** - **Robustness:** The coefficient stability (CV < 5%) proves the model is immune to multicollinearity. 
- **Physical Coherence:** The residuals follow a near-perfect Gaussian distribution (Mean Bias ~0.81g), indicating that all deterministic signal has been captured.

- **Limitations:**

- **Interpretability:** Because the model operates in a transformed space, we cannot say "1 liter adds X grams" directly. We must use the inverse transformation to get real values.
- **Scope:** The slight residual spread at the high end (>350g/km) suggests that for extreme heavy-duty vehicles, a separate model or additional features (like vehicle weight) might be required.

### 6. Deployment:
---

In [0]:
# DEPLOY PACKAGE: We save everything in a dictionary to ensure integrity.
production_bundle = {
    'model': model, 
    'pt_X': X_scaler, 
    'pt_y': y_scaler,
}

# Saved in a single "pickle" file
joblib.dump(production_bundle, './artifacts/co2_pipeline_v2.pkl')

# Return
print('✅ Complete pipeline saved successfully!')

In [0]:
def predict_emission(engine_size, fuel_consumption):
    
    """
    Performs full inference with Yeo-Johnson pre- and post-processing.
    """
    # 1. Loading (Load the Complete Package)
    bundle = joblib.load('./artifacts/co2_pipeline_v2.pkl')
    model_loaded = bundle['model']
    pt_X_loaded = bundle['pt_X']
    pt_y_loaded = bundle['pt_y']

    # 2. Input Data Engineering
    input_data = pd.DataFrame(
        [[engine_size, fuel_consumption]],
        columns = ['ENGINESIZE', 'FUELCONSUMPTION_COMB']
    )

    # 3. Pre-processing
    input_transformed = pt_X_loaded.transform(input_data)

    # 4. Prediction
    prediction_transformed = model_loaded.predict(input_transformed)

    # 5. Post-processing (Reducing to Grams of CO2)
    prediction_real  = pt_y_loaded.inverse_transform(prediction_transformed.reshape(-1, 1))

    result_g_km = prediction_real[0][0]

    return result_g_km

# --- FINAL TEST (User Acceptance Test) ---

# Scenario: 2.0L car getting 8.5 L/100km
engine = 2.0
consumption = 8.5

try:
    prediction = predict_emission(engine, consumption)

    print(f'--- INFERENCE REPORT ---')
    print(f'Engine: {engine} L')
    print(f'Consumption: {consumption} L/100km')
    print(f'Predicted Emission: {prediction:.2f} g/km')

except Exception as e:
    print(f'Deployment error: {e}')